# Data Error Generation

This notebook generates a dataset with various data quality issues to test the data ingestion pipeline.
We will use the `gym_members_exercise_tracking.csv` dataset and introduce 7 types of errors.

In [ ]:
import pandas as pd
import numpy as np
import random
import os

# Load the dataset
INPUT_FILE = '../gym_members_exercise_tracking.csv'
OUTPUT_DIR = '../data/raw_data_with_errors'
os.makedirs(OUTPUT_DIR, exist_ok=True)

df = pd.read_csv(INPUT_FILE)
print(f"Original dataset shape: {df.shape}")
df.head()

## Define Error Injection Functions

In [ ]:
def inject_missing_values(df, column, probability=0.1):
    """Injects NaN values into a column."""
    df = df.copy()
    mask = np.random.rand(len(df)) < probability
    df.loc[mask, column] = np.nan
    return df

def inject_unknown_category(df, column, new_value, probability=0.1):
    """Injects an unknown categorical value."""
    df = df.copy()
    mask = np.random.rand(len(df)) < probability
    df.loc[mask, column] = new_value
    return df

def inject_out_of_range(df, column, min_val=None, max_val=None, probability=0.1):
    """Injects values outside the expected range."""
    df = df.copy()
    mask = np.random.rand(len(df)) < probability
    if min_val is not None:
        df.loc[mask, column] = min_val - abs(np.random.randn(sum(mask)) * 10)
    elif max_val is not None:
        df.loc[mask, column] = max_val + abs(np.random.randn(sum(mask)) * 10)
    return df

def inject_type_mismatch(df, column, probability=0.1):
    """Injects string values into a numerical column."""
    df = df.copy()
    mask = np.random.rand(len(df)) < probability
    df.loc[mask, column] = "INVALID_TYPE"
    return df

def inject_negative_value(df, column, probability=0.1):
    """Injects negative values where positive are expected."""
    df = df.copy()
    mask = np.random.rand(len(df)) < probability
    df.loc[mask, column] = df.loc[mask, column] * -1
    return df

def remove_column(df, column):
    """Removes a required column."""
    return df.drop(columns=[column])

def inject_duplicates(df, probability=0.1):
    """Injects duplicate rows."""
    n_dupes = int(len(df) * probability)
    dupes = df.sample(n=n_dupes, replace=True)
    return pd.concat([df, dupes], ignore_index=True)

## Generate Error Files

We will generate separate files for different error types to test the validation logic.

In [ ]:
# 1. Missing Values in 'Age'
df_missing = inject_missing_values(df, 'Age', 0.2)
df_missing.to_csv(f"{OUTPUT_DIR}/error_missing_values.csv", index=False)

# 2. Unknown Category in 'Gender' (e.g., 'Alien')
df_unknown_cat = inject_unknown_category(df, 'Gender', 'Alien', 0.2)
df_unknown_cat.to_csv(f"{OUTPUT_DIR}/error_unknown_category.csv", index=False)

# 3. Out of Range 'Heart_Rate' (e.g., 300+)
df_range = inject_out_of_range(df, 'Heart_Rate', max_val=250, probability=0.2)
df_range.to_csv(f"{OUTPUT_DIR}/error_out_of_range.csv", index=False)

# 4. Type Mismatch in 'Body_Temp' (String instead of float)
df_type = inject_type_mismatch(df, 'Body_Temp', 0.2)
df_type.to_csv(f"{OUTPUT_DIR}/error_type_mismatch.csv", index=False)

# 5. Negative 'Duration' (Logic error)
df_negative = inject_negative_value(df, 'Duration', 0.2)
df_negative.to_csv(f"{OUTPUT_DIR}/error_negative_value.csv", index=False)

# 6. Missing Column 'BMI' (Schema error)
df_no_col = remove_column(df, 'BMI')
df_no_col.to_csv(f"{OUTPUT_DIR}/error_missing_column.csv", index=False)

# 7. Duplicates
df_dupes = inject_duplicates(df, 0.3)
df_dupes.to_csv(f"{OUTPUT_DIR}/error_duplicates.csv", index=False)

# 8. Mixed Errors (Chaos)
df_chaos = df.copy()
df_chaos = inject_missing_values(df_chaos, 'Age', 0.1)
df_chaos = inject_unknown_category(df_chaos, 'Gender', 'Robot', 0.1)
df_chaos = inject_negative_value(df_chaos, 'Duration', 0.1)
df_chaos.to_csv(f"{OUTPUT_DIR}/error_mixed_chaos.csv", index=False)

print(f"Generated 8 error files in {OUTPUT_DIR}")